# TiO2 Results Analysis - Create Publication Plots

This notebook creates publication-quality plots from the computed metrics.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from analysis.plotting import (
    plot_calibration,
    plot_sharpness,
    plot_performance_comparison,
    plot_residuals_vs_uncertainty
)

# Set plotting style
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'figure.dpi': 150
})
sns.set_context("talk", font_scale=1.2)

In [ ]:
# Load results
df = pd.read_csv('results/uq_metrics_Test.csv')

# Create figures directory
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

print(f"Loaded {len(df)} experiments")
df.head()

## Performance Comparison Plots

In [ ]:
# Plot performance metrics
metrics_to_plot = ['mae', 'rmse', 'maxerr']
methods = df['Method'].unique()
sizes = df['Size'].unique()

figs = plot_performance_comparison(
    df,
    metrics=metrics_to_plot,
    methods=methods,
    sizes=sizes,
    figsize=(12, 6),
    save_path=str(fig_dir / 'performance_{metric}.png')
)

# Display
for metric, fig in figs.items():
    plt.show()

## UQ Quality Plots

In [ ]:
# Plot UQ metrics (only for methods with uncertainty)
uq_methods = [m for m in methods if m != 'nn']  # Exclude deterministic NN
df_uq = df[df['Method'].isin(uq_methods)]

if len(df_uq) > 0:
    uq_metrics = ['overlap', 'sharp', 'nll']
    
    figs = plot_performance_comparison(
        df_uq,
        metrics=uq_metrics,
        methods=uq_methods,
        sizes=sizes,
        figsize=(12, 6),
        save_path=str(fig_dir / 'uq_quality_{metric}.png')
    )
    
    for metric, fig in figs.items():
        plt.show()

## Sharpness Distribution

In [ ]:
# Plot sharpness distribution
if 'sharp' in df_uq.columns:
    fig = plot_sharpness(
        df_uq,
        methods=uq_methods,
        sizes=sizes,
        figsize=(12, 6),
        save_path=str(fig_dir / 'sharpness_distribution.png')
    )
    plt.show()

## Summary Table

In [ ]:
# Create summary table
summary = df.groupby(['Method', 'Size'])[['mae', 'rmse', 'overlap', 'nll']].agg(['mean', 'std'])
summary = summary.round(4)

# Save to CSV
summary.to_csv(fig_dir / 'summary_table.csv')

# Display
print("\nSummary Statistics:")
summary

## Best Performing Models

In [ ]:
# Find best models for each metric
print("\nBest Models (by MAE):")
for size in sizes:
    df_size = df[df['Size'] == size]
    best_idx = df_size['mae'].idxmin()
    best = df_size.loc[best_idx]
    print(f"\n{size}:")
    print(f"  Method: {best['Method']}")
    print(f"  Run: {best['Run']}")
    print(f"  MAE: {best['mae']:.4f}")
    print(f"  RMSE: {best['rmse']:.4f}")